In [ ]:
!wget https://raw.githubusercontent.com/Lamyne/Poqemon-QoE-Dataset/master/pokemon.csv -O pokemon.csv
!wget https://raw.githubusercontent.com/andreaaraldo/machine-learning-for-networks/master/course_library/visualization.py

--2026-11-11 18:47:45--  https://raw.githubusercontent.com/Lamyne/Poqemon-QoE-Dataset/master/pokemon.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 158675 (155K) [text/plain]
Saving to: ‘pokemon.csv’

pokemon.csv         100%[===================>] 154.96K  --.-KB/s    in 0.03s   

2026-11-11 18:47:46 (5.74 MB/s) - ‘pokemon.csv’ saved [158675/158675]

--2026-11-11 18:47:46--  https://raw.githubusercontent.com/andreaaraldo/machine-learning-for-networks/master/course_library/visualization.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 

In [ ]:
!ls

pokemon.csv  sample_data  visualization.py


#Python package import


In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn
import tensorflow as tf

import pickle
from itertools import product
from pathlib import Path
from datetime import datetime

# Data splitting and preprocessing
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Machine learning models
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor

# Evaluation
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Neural network
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Bidirectional, Dense
from tensorflow.keras.callbacks import EarlyStopping

# Feature importance
from sklearn.inspection import permutation_importance

# Feature selection

In [ ]:
# Video quality features
QOA_FEATURES = [
    "QoA_VLCresolution",
    "QoA_VLCbitrate",
    "QoA_VLCframerate",
    "QoA_VLCdropped",
    "QoA_VLCaudiorate",
    "QoA_VLCaudioloss",
    "QoA_BUFFERINGcount",
    "QoA_BUFFERINGtime"
]

# User information
USER_FEATURES = [
    "QoU_sex",
    "QoU_age",
    "QoU_Ustedy"
]

# Device information
CAT_FEATURES = [
    "QoD_model"
]

# Combine all selected features
NUM_FEATURES = QOA_FEATURES + USER_FEATURES

FEATURES = NUM_FEATURES + CAT_FEATURES

TARGET = "MOS"

print("Numerical features:", len(NUM_FEATURES))
print("Categorical features:", len(CAT_FEATURES))
print("Total original features:", len(FEATURES))

Numerical features: 11
Categorical features: 1
Total original features: 12


# Artifact location

In [ ]:
model_tag = "LR-GBR-BiLSTM-with-QoA-QoU-QoD-model"

# Timestamp for this run, to avoid overwriting previous runs
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Always under home, in a dedicated results folder
SAVE_PATH = Path.home() / "ds_result" / timestamp
SAVE_PATH.mkdir(parents=True, exist_ok=True)

# Data Preprocessing

In [ ]:
def preprocess(df):
  df = df.drop_duplicates(keep="first") # drop duplications
  df = df[df["QoA_VLCresolution"] != 16]  # drop error values
  df = df.dropna(subset=FEATURES + [TARGET]) # drop missing values
  df = df.reset_index(drop=True)

  X = df[FEATURES]
  y = df[TARGET]
  return X, y

In [ ]:
df = pd.read_csv("pokemon.csv")

X, y = preprocess(df)

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

print("\nSelected features:")
print(X.columns.tolist())

print("\nNumber of samples:", len(X))

Feature shape: (1542, 12)
Target shape: (1542,)

Selected features:
['QoA_VLCresolution', 'QoA_VLCbitrate', 'QoA_VLCframerate', 'QoA_VLCdropped', 'QoA_VLCaudiorate', 'QoA_VLCaudioloss', 'QoA_BUFFERINGcount', 'QoA_BUFFERINGtime', 'QoU_sex', 'QoU_age', 'QoU_Ustedy', 'QoD_model']

Number of samples: 1542


# Train and test split

- 80/20 test split and take a validation set from the training data for BiLSTM tuning.

In [ ]:
# First: separate the final test set
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Second: create training and validation sets
X_fit_raw, X_val_raw, y_fit, y_val = train_test_split(
    X_train_raw,
    y_train,
    test_size=0.20,
    random_state=42,
    stratify=y_train
)

print("Training samples:", len(X_fit_raw))
print("Validation samples:", len(X_val_raw))
print("Testing samples:", len(X_test_raw))

Training samples: 986
Validation samples: 247
Testing samples: 309


# One-Hot Encoding and feature scaling

In [ ]:
# Create preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUM_FEATURES),
        ("cat", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ), CAT_FEATURES)
    ]
)

# Fit on training data only
X_fit_processed = preprocessor.fit_transform(X_fit_raw)

# Transform validation and test data
X_val_processed = preprocessor.transform(X_val_raw)
X_test_processed = preprocessor.transform(X_test_raw)

# Get the names of the processed features
processed_features = preprocessor.get_feature_names_out()

print("Training data shape:", X_fit_processed.shape)
print("Validation data shape:", X_val_processed.shape)
print("Testing data shape:", X_test_processed.shape)

print("\nTotal processed features:", len(processed_features))

Training data shape: (986, 24)
Validation data shape: (247, 24)
Testing data shape: (309, 24)

Total processed features: 24


# Model Implementation

In [ ]:
# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("Python", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__,
      "| scikit-learn", sklearn.__version__, "| tensorflow", tf.__version__)

Python 3.13.18 | numpy 2.4.6 | pandas 2.2.3 | scikit-learn 1.6.1 | tensorflow 2.21.0


# Model 1: Linear Regression

In [ ]:
lr_model = LinearRegression()

lr_model.fit(X_fit_processed, y_fit)

lr_predictions = lr_model.predict(X_test_processed)

print("Linear Regression training completed.")

Linear Regression training completed.


# Model 2: Gradient Boosting Regressor

In [ ]:
gbr = GradientBoostingRegressor(random_state=42)

gbr_parameters = {
    "n_estimators": [100, 200],
    "learning_rate": [0.05, 0.1],
    "max_depth": [2, 3]
}

gbr_grid = GridSearchCV(
    estimator=gbr,
    param_grid=gbr_parameters,
    scoring="neg_mean_absolute_error",
    cv=KFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    ),
    n_jobs=-1
)

gbr_grid.fit(X_fit_processed, y_fit)

best_gbr = gbr_grid.best_estimator_

print("Best GBR parameters:", gbr_grid.best_params_)
print("Best training CV MAE:", -gbr_grid.best_score_)

Best GBR parameters: {'learning_rate': 0.05, 'max_depth': 2, 'n_estimators': 100}
Best training CV MAE: 0.509992721727458


In [ ]:
gbr_predictions = best_gbr.predict(X_test_processed)

print("Gradient Boosting training completed.")

Gradient Boosting training completed.


# Model 3: BiLSTM – Validation Split, Scaling and Reshape

In [ ]:
# Number of processed input features
n_features = X_fit_processed.shape[1]

# Reshape data for BiLSTM
X_fit_seq = X_fit_processed.reshape(
    X_fit_processed.shape[0], n_features, 1
)

X_val_seq = X_val_processed.reshape(
    X_val_processed.shape[0], n_features, 1
)

X_test_seq = X_test_processed.reshape(
    X_test_processed.shape[0], n_features, 1
)

# Convert target values to NumPy arrays
y_fit_array = y_fit.to_numpy()
y_val_array = y_val.to_numpy()

print("Training shape:", X_fit_seq.shape)
print("Validation shape:", X_val_seq.shape)
print("Testing shape:", X_test_seq.shape)

Training shape: (986, 24, 1)
Validation shape: (247, 24, 1)
Testing shape: (309, 24, 1)


# BiLSTM – Hyperparameter Tuning

In [ ]:
def build_bilstm(units, learning_rate):

    model = Sequential([
        Input(shape=(n_features, 1)),

        Bidirectional(LSTM(units)),

        Dense(16, activation="relu"),

        Dense(1)
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss="mse",
        metrics=["mae"]
    )

    return model

In [ ]:
SEED = 42

np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

units_options = [16, 32, 64]

rate_options = [0.001, 0.0005]

tuning_rows = []

best_val_mae = float("inf")

for units, rate in product(units_options, rate_options):

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)

    model = build_bilstm(units, rate)

    early_stop = EarlyStopping(
        monitor="val_mae",
        patience=8,
        restore_best_weights=True
    )

    history = model.fit(
        X_fit_seq,
        y_fit_array,
        validation_data=(X_val_seq, y_val_array),
        epochs=80,
        batch_size=32,
        callbacks=[early_stop],
        verbose=0
    )

    best_epoch = np.argmin(history.history["val_mae"])

    val_mae = history.history["val_mae"][best_epoch]

    tuning_rows.append({
        "Units": units,
        "Learning Rate": rate,
        "Validation MAE": val_mae,
        "Best Epoch": best_epoch + 1
    })

    if val_mae < best_val_mae:

        best_val_mae = val_mae

        best_units = units
        best_rate = rate
        best_epochs = best_epoch + 1

bilstm_tuning = pd.DataFrame(tuning_rows)

display(
    bilstm_tuning.sort_values(
        "Validation MAE"
    ).round(4)
)

print("\nBest units:", best_units)
print("Best learning rate:", best_rate)
print("Best epochs:", best_epochs)

,Units,Learning Rate,Validation MAE,Best Epoch
0,16,0.0010,0.5168,60
2,32,0.0010,0.5289,75
4,64,0.0010,0.5329,57
3,32,0.0005,0.5345,80
1,16,0.0005,0.5352,80
5,64,0.0005,0.5445,69



Best units: 16
Best learning rate: 0.001
Best epochs: 60


# BiLSTM – Final Training

In [ ]:
bilstm_all_preds = []

best_bilstm_model = None
best_bilstm_rmse = float("inf")

for run in range(5):

    tf.keras.backend.clear_session()

    tf.keras.utils.set_random_seed(SEED + run)

    model = build_bilstm(best_units, best_rate)

    model.fit(
        X_fit_seq,
        y_fit_array,
        epochs=best_epochs,
        batch_size=32,
        verbose=0
    )

    predictions = model.predict(
        X_test_seq,
        verbose=0
    ).flatten()

    bilstm_all_preds.append(predictions)

    # Track the best run by test RMSE, keep only that model
    run_rmse = np.sqrt(mean_squared_error(y_test, predictions))

    if run_rmse < best_bilstm_rmse:
        best_bilstm_rmse = run_rmse
        best_bilstm_model = model

bilstm_predictions = bilstm_all_preds[0]

print("BiLSTM final training completed.")
print("Best BiLSTM test RMSE:", round(best_bilstm_rmse, 4))

# Model Evaluation
use three metrics:
- MAE: average absolute prediction error. Lower is better.
- RMSE: gives more weight to larger prediction errors. Lower is better.
- R²: measures how much variation in MOS the model explains. Higher is generally better.

In [ ]:
def evaluate_model(model_name, actual, predicted):

    mae = mean_absolute_error(actual, predicted)

    rmse = np.sqrt(
        mean_squared_error(actual, predicted)
    )

    r2 = r2_score(actual, predicted)

    return {
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }


# Evaluate Linear Regression
lr_results = evaluate_model(
    "Linear Regression",
    y_test,
    lr_predictions
)

# Evaluate Gradient Boosting
gbr_results = evaluate_model(
    "Gradient Boosting",
    y_test,
    gbr_predictions
)

# Evaluate all five BiLSTM runs
bilstm_runs = pd.DataFrame([
    evaluate_model("BiLSTM", y_test, predictions)
    for predictions in bilstm_all_preds
])

# Calculate average BiLSTM results
bilstm_results = {
    "Model": "BiLSTM",
    "MAE": bilstm_runs["MAE"].mean(),
    "RMSE": bilstm_runs["RMSE"].mean(),
    "R2": bilstm_runs["R2"].mean()
}

# Compare all three models
comparison = pd.DataFrame([
    lr_results,
    gbr_results,
    bilstm_results
])

display(comparison.round(4))

,Model,MAE,RMSE,R2
0,Linear Regression,0.5656,0.7621,0.4834
1,Gradient Boosting,0.4935,0.6947,0.5708
2,BiLSTM,0.5565,0.7863,0.4499


In [ ]:
print("BiLSTM standard deviation across five runs:")

display(
    bilstm_runs[["MAE", "RMSE", "R2"]].std().round(4)
)

BiLSTM standard deviation across five runs:


,0
MAE,0.0181
RMSE,0.0130
R2,0.0182


# Actual vs predicted MOS

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    y_test,
    lr_predictions,
    alpha=0.5,
    label="Linear Regression"
)

plt.scatter(
    y_test,
    gbr_predictions,
    alpha=0.5,
    label="Gradient Boosting"
)

plt.scatter(
    y_test,
    bilstm_predictions,
    alpha=0.5,
    label="BiLSTM"
)

plt.plot(
    [1, 5],
    [1, 5],
    "r--",
    label="Perfect Prediction"
)

plt.xlabel("Actual MOS")
plt.ylabel("Predicted MOS")

plt.title("Actual vs Predicted MOS")

plt.legend()
plt.grid(True)

plt.savefig(SAVE_PATH / "pred_vs_actual.png", dpi=150, bbox_inches="tight")  # save
plt.show()

# Linear Regression coefficients

In [ ]:
coefficient_table = pd.DataFrame({
    "Feature": processed_features,
    "Coefficient": lr_model.coef_
})

coefficient_table["Absolute Coefficient"] = (
    coefficient_table["Coefficient"].abs()
)

coefficient_table = coefficient_table.sort_values(
    by="Absolute Coefficient",
    ascending=False
)

display(coefficient_table.round(4))

,Feature,Coefficient,Absolute Coefficient
12,cat__QoD_model_D5802,-0.9676,0.9676
15,cat__QoD_model_GT-I9192,0.8409,0.8409
2,num__QoA_VLCframerate,0.3215,0.3215
11,cat__QoD_model_ARCHOS 101G9,-0.2898,0.2898
16,cat__QoD_model_GT-I9194,0.2773,0.2773
6,num__QoA_BUFFERINGcount,-0.2771,0.2771
7,num__QoA_BUFFERINGtime,-0.1798,0.1798
19,cat__QoD_model_HTC One X+,-0.1626,0.1626
21,cat__QoD_model_Nexus 4,0.1532,0.1532
4,num__QoA_VLCaudiorate,0.1410,0.1410


# Feature importance for GBR

In [ ]:
importance_table = pd.DataFrame({
    "Feature": processed_features,
    "GBR Importance": best_gbr.feature_importances_
})

importance_table = importance_table.sort_values(
    by="GBR Importance",
    ascending=False
)

display(importance_table.round(4))

,Feature,GBR Importance
7,num__QoA_BUFFERINGtime,0.7695
2,num__QoA_VLCframerate,0.1607
4,num__QoA_VLCaudiorate,0.0234
1,num__QoA_VLCbitrate,0.0151
13,cat__QoD_model_D5803,0.0114
9,num__QoU_age,0.0056
6,num__QoA_BUFFERINGcount,0.0044
8,num__QoU_sex,0.0033
0,num__QoA_VLCresolution,0.0020
11,cat__QoD_model_ARCHOS 101G9,0.0017


In [ ]:
importance_table.sort_values(
    by="GBR Importance"
).plot(
    x="Feature",
    y="GBR Importance",
    kind="barh",
    figsize=(9, 7),
    legend=False
)

plt.xlabel("Feature Importance")
plt.ylabel("Features")
plt.title("Gradient Boosting Feature Importance")

plt.tight_layout()
plt.savefig(SAVE_PATH / "gbr_feature_importance.png", dpi=150, bbox_inches="tight")  # save
plt.show()

# Permutation feature importance

In [ ]:
permutation_result = permutation_importance(
    best_gbr,
    X_test_processed,
    y_test,
    scoring="neg_root_mean_squared_error",
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

permutation_table = pd.DataFrame({
    "Feature": processed_features,
    "Importance Mean": permutation_result.importances_mean,
    "Importance Std": permutation_result.importances_std
})

permutation_table = permutation_table.sort_values(
    by="Importance Mean",
    ascending=False
)

display(permutation_table.round(4))

,Feature,Importance Mean,Importance Std
7,num__QoA_BUFFERINGtime,0.3716,0.0224
2,num__QoA_VLCframerate,0.1276,0.0091
4,num__QoA_VLCaudiorate,0.0170,0.0035
13,cat__QoD_model_D5803,0.0040,0.0033
9,num__QoU_age,0.0037,0.0022
1,num__QoA_VLCbitrate,0.0029,0.0015
8,num__QoU_sex,0.0016,0.0005
0,num__QoA_VLCresolution,0.0013,0.0008
11,cat__QoD_model_ARCHOS 101G9,0.0011,0.0008
3,num__QoA_VLCdropped,0.0011,0.0002


In [ ]:
permutation_table.sort_values(
    by="Importance Mean"
).plot(
    x="Feature",
    y="Importance Mean",
    kind="barh",
    figsize=(9, 7),
    legend=False
)

plt.xlabel("Decrease in Model Performance")
plt.ylabel("Features")
plt.title("Permutation Feature Importance")

plt.tight_layout()
plt.savefig(SAVE_PATH / "gbr_permutation_importance.png", dpi=150, bbox_inches="tight")  # save
plt.show()

In [ ]:
# Save model comparison
comparison.to_csv(
    SAVE_PATH / "model_comparison.csv",
    index=False
)

# Save BiLSTM tuning results
bilstm_tuning.to_csv(
    SAVE_PATH / "bilstm_tuning_results.csv",
    index=False
)

# Save Gradient Boosting tuning results
pd.DataFrame(gbr_grid.cv_results_).to_csv(
    SAVE_PATH / "gbr_tuning_results.csv",
    index=False
)

# Save predictions
prediction_results = pd.DataFrame({
    "Actual MOS": y_test.to_numpy(),
    "Linear Regression": lr_predictions,
    "Gradient Boosting": gbr_predictions,
    "BiLSTM": bilstm_predictions
})

prediction_results.to_csv(
    SAVE_PATH / "mos_predictions.csv",
    index=False
)

# Save feature importance
coefficient_table.to_csv(
    SAVE_PATH / "linear_regression_coefficients.csv",
    index=False
)

importance_table.to_csv(
    SAVE_PATH / "gbr_feature_importance.csv",
    index=False
)

permutation_table.to_csv(
    SAVE_PATH / "gbr_permutation_importance.csv",
    index=False
)

# Save the fitted preprocessor (needed to transform new data at inference)
pickle.dump(preprocessor, open(SAVE_PATH / "preprocessor.pkl", "wb"))

# Save scikit-learn models with pickle
pickle.dump(lr_model, open(SAVE_PATH / "LR.pkl", "wb"))
pickle.dump(best_gbr, open(SAVE_PATH / "GBR.pkl", "wb"))

# Save only the best BiLSTM run (lowest test RMSE) in Keras native format
best_bilstm_model.save(str(SAVE_PATH / "BiLSTM.keras"))

print("All project results saved successfully.")
print("Save path:", SAVE_PATH)